<a href="https://colab.research.google.com/github/HarishGireesan/Polyformer_gpe24/blob/main/PDF_Radius_Extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# 1. Setup: Install necessary libraries (if not already installed in Colab)
# The '!' tells Colab to run this command in the system, installing the libraries.
!pip install pypdf pandas

import pypdf
import re
import pandas as pd
from google.colab import files
import io
import zipfile
import os
import shutil

# 2. Define the extraction function
def extract_r_value(file_content, filename):
    """
    Extracts the 'r' value (radius) from the raw text content of the PDF.

    The extraction is based on the format observed in the provided file:
    "r [number] µm"
    """
    try:
        # Read the file content from the byte buffer
        reader = pypdf.PdfReader(io.BytesIO(file_content))

        # We assume the relevant table data is on the first page
        page = reader.pages[0]
        text = page.extract_text()

        # Regular Expression to find the 'r' value in the format "r [number] µm"
        match = re.search(r'r\s*([\d\.]+)\s*µm', text, re.IGNORECASE)

        if match:
            return float(match.group(1))

        # If the pattern is not found
        # print(f"Warning: 'r' value not found in {filename}.") # Removed verbose print
        return None

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        return None

# 3. File Upload and Mapping
# Check if files have already been processed in this session
if 'pdf_files_to_process' not in locals() or not pdf_files_to_process:
    print("Please upload your ZIP file containing all the PDF reports now.")
    # This is where the upload prompt is triggered.
    uploaded = files.upload()

    # --- NEW ZIP HANDLING LOGIC ---
    if not uploaded:
        print("No file was uploaded. Exiting.")
        # exit() # Do not exit, allow code to run with potentially empty pdf_files_to_process

    zip_filename = list(uploaded.keys())[0]
    zip_content = uploaded[zip_filename]
    temp_dir = 'pdf_reports'
    pdf_files_to_process = {}

    print(f"\nUnzipping {zip_filename}...")

    # Create a temporary directory to extract files
    os.makedirs(temp_dir, exist_ok=True)

    try:
        with zipfile.ZipFile(io.BytesIO(zip_content), 'r') as z:
            # Extract only PDF files and store their contents
            for member in z.infolist():
                # Skip directories and non-pdf files
                if not member.is_dir() and member.filename.lower().endswith('.pdf'):
                    # Clean up the filename for display (remove leading directory paths)
                    clean_filename = os.path.basename(member.filename)

                    # Read the file content directly from the zip stream
                    with z.open(member) as pdf_file:
                        pdf_files_to_process[clean_filename] = pdf_file.read()

        print(f"Found {len(pdf_files_to_process)} PDF files to process.")

    except zipfile.BadZipFile:
        print(f"Error: The file {zip_filename} is not a valid zip file.")
        # Exit or proceed with empty list if the zip file is bad
        pdf_files_to_process = {}
    except Exception as e:
        print(f"An error occurred during zip processing: {e}")
        pdf_files_to_process = {}
else:
    print("Using already uploaded files.")
    # Clean up the temporary directory from previous runs if it exists
    temp_dir = 'pdf_reports'
    shutil.rmtree(temp_dir, ignore_errors=True)
    os.makedirs(temp_dir, exist_ok=True) # Recreate the directory for potential future use, though not strictly needed if not extracting


# 4. Processing the uploaded files (now extracted from zip)
data = []

# Iterate over the extracted PDF file contents
for filename, content in pdf_files_to_process.items():
    r_value = extract_r_value(content, filename)

    # Extract "Hauptversuche_XX" and "Kante_Y" or "Graphit/Graphite" from filename
    # Adjusted regex to match the new filename format "Hauptversuche_afterexperiment_XX"
    hauptversuche_match = re.search(r'Hauptversuche_afterexperiment_(\d+)', filename)
    kante_match = re.search(r'Kante_(\d+)', filename)
    graphite_match = re.search(r'(Graphit|Graphite)', filename, re.IGNORECASE)

    hauptversuche_num = None
    material_type = "Other" # Default to Other if no specific type found

    if hauptversuche_match:
        hauptversuche_num = int(hauptversuche_match.group(1)) # Convert to int for sorting
        if kante_match:
            kante_num = int(kante_match.group(1)) # Convert to int for sorting
            material_type = f'kante {kante_num}'
        elif graphite_match:
            material_type = 'graphite'
    elif graphite_match:
         material_type = 'graphite' # Fallback for graphite files without Hauptversuche number

    # Always append data, even if Hauptversuche is None, to ensure columns exist
    data.append({
        'File Name': filename,
        'Hauptversuche': hauptversuche_num,
        'Material Type': material_type,
        'r (\u00B5m)': r_value
    })

# Create initial DataFrame from extracted data
df = pd.DataFrame(data)

# --- Data Transformation for Desired Output ---
# Filter out rows that don't have a Hauptversuche number unless it's a graphite file with one
# For the pivoted table, let's strictly use entries with a Hauptversuche number for rows
hauptversuche_pivot_data = df[df['Hauptversuche'].notna()].copy()


# Pivot the table to get Kante values as columns for each Hauptversuche
pivot_df = hauptversuche_pivot_data.pivot_table(
    index='Hauptversuche',
    columns='Material Type',
    values='r (\u00B5m)',
    aggfunc='first' # Use 'first' as there should be only one value per Hauptversuche/Kante combination expected
).reset_index()


# Rename columns for clarity in the final output
pivot_df.columns.name = None # Remove the columns name
pivot_df = pivot_df.rename(columns={
    'Hauptversuche': 'Hauptversuche',
    'kante 1': 'kante 1 (r in µm)',
    'kante 2': 'kante 2 (r in µm)',
    'kante 3': 'kante 3 (r in µm)',
    'kante 4': 'kante 4 (r in µm)',
    'graphite': 'graphite (r in µm)' # Include graphite if it has a Hauptversuche number
})

# Ensure all kante columns are present even if no data for them
all_kante_columns = [f'kante {i} (r in µm)' for i in range(1, 5)]
required_columns = ['Hauptversuche'] + all_kante_columns
if 'graphite (r in µm)' in pivot_df.columns:
    required_columns.append('graphite (r in µm)')

# Reindex to ensure all required columns are present and in order
final_pivot_df = pivot_df.reindex(columns=required_columns)


# Display the transformed DataFrame
display(final_pivot_df)

print("\n" + "="*50)
print("             Transformed Extracted Radius (r) Table")
print("="*50)
print(final_pivot_df.to_markdown(index=False))


# Clean up the temporary directory
# Only remove if the files were just uploaded
if 'uploaded' in locals() and uploaded:
  shutil.rmtree(temp_dir, ignore_errors=True)
# --- END NEW ZIP HANDLING LOGIC ---

Using already uploaded files.


,Hauptversuche,kante 1 (r in µm),kante 2 (r in µm),kante 3 (r in µm),kante 4 (r in µm)
0,1,28.7452,16.1309,10.8971,13.3173
1,6,50.8914,58.3078,52.3829,52.1129
2,7,11.6407,9.9918,24.0433,10.1639
3,8,29.8903,34.8572,34.1025,31.9895
4,9,55.4935,53.1533,57.8855,52.7781
5,10,56.2085,54.7251,53.5778,58.5244
6,11,33.5568,33.9081,33.5811,30.7969
7,12,35.4954,35.7459,34.7085,36.2092
8,13,16.1626,17.7866,20.9351,20.5224
9,14,75.6095,81.1515,81.6938,77.3371



             Transformed Extracted Radius (r) Table
|   Hauptversuche |   kante 1 (r in µm) |   kante 2 (r in µm) |   kante 3 (r in µm) |   kante 4 (r in µm) |
|----------------:|--------------------:|--------------------:|--------------------:|--------------------:|
|               1 |             28.7452 |             16.1309 |             10.8971 |             13.3173 |
|               6 |             50.8914 |             58.3078 |             52.3829 |             52.1129 |
|               7 |             11.6407 |              9.9918 |             24.0433 |             10.1639 |
|               8 |             29.8903 |             34.8572 |             34.1025 |             31.9895 |
|               9 |             55.4935 |             53.1533 |             57.8855 |             52.7781 |
|              10 |             56.2085 |             54.7251 |             53.5778 |             58.5244 |
|              11 |             33.5568 |             33.9081 |             33.5811

In [15]:
print (df)

                                             File Name Hauptversuche  \
0    AF5_RC_poliert_Hauptversuche_afterexperiment_1...          None   
1    AF5_RC_poliert_Hauptversuche_afterexperiment_1...          None   
2    AF5_RC_poliert_Hauptversuche_afterexperiment_1...          None   
3    AF5_RC_poliert_Hauptversuche_afterexperiment_1...          None   
4    AF5_RC_poliert_Hauptversuche_afterexperiment_1...          None   
..                                                 ...           ...   
167  AF5_RC_poliert_Hauptversuche_afterexperiment_8...          None   
168  AF5_RC_poliert_Hauptversuche_afterexperiment_9...          None   
169  AF5_RC_poliert_Hauptversuche_afterexperiment_9...          None   
170  AF5_RC_poliert_Hauptversuche_afterexperiment_9...          None   
171  AF5_RC_poliert_Hauptversuche_afterexperiment_9...          None   

    Material Type   r (µm)  
0           Other  56.2085  
1           Other  54.7251  
2           Other  53.5778  
3           Other  

In [24]:
# 6. Save the final results to an Excel file
try:
    excel_filename = 'extracted_r_values.xlsx'
    # Save the full DataFrame instead of the filtered one
    final_pivot_df.to_excel(excel_filename, index=False)
    print(f"\nSuccessfully saved the extracted data to '{excel_filename}'")

    # Offer to download the file
    files.download(excel_filename)

except Exception as e:
    print(f"\nError saving the Excel file: {e}")


Successfully saved the extracted data to 'extracted_r_values.xlsx'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
# Inspect the content of one of the PDF files to understand the format
# You can change the filename here to inspect a different file if needed
if pdf_files_to_process:
    sample_filename = list(pdf_files_to_process.keys())[0]
    sample_content = pdf_files_to_process[sample_filename]

    # Read the file content from the byte buffer
    reader = pypdf.PdfReader(io.BytesIO(sample_content))

    # We assume the relevant table data is on the first page
    page = reader.pages[0]
    text = page.extract_text()

    print(text)
else:
    print("No PDF files found in the uploaded zip.")

IF-EdgeMasterModule Measurement Report 
Schneidkantenmessung 
Referenztyp: 
Anzahl der extrahierten Profile: 
Kantenprofiltyp: 
Datum der Messung: 
Prüfer: 
Standard Straight Edge 20x 
50 
Keine Fase 
05.09.2025 16:23:58 
Administrator 
Name Wert [u] Beschreibung 
r 56.2085 µm Mittlerer Radius der Durchschnittskante 
α -0.4211 ° Freiwinkel 
β 89.8060 ° Keilwinkel 
γ 0.6151 ° Spanwinkel 
Sα 55.5108 µm Dist. Apex zum Ende der Freiflächenrundheit (früher: a) 
Sγ 122.6013 µm Dist. Apex zum Ende der Spanflächenrundheit (früher: b) 
K 2.2086 Symmetrie der Schneidkante 
∆r 31.3268 µm Min. Dist. der Kante zum Apex (früher: S) 
W 62.6852 µm Kantenbreite 
Ecq 0.2524 µm Formabweichung des Kreises (RMS) 
Form Trompete Erwartete Krümmung 
Alicona Imaging GmbH 
Dr.-Auner Strasse 21a 
A-8074 Raaba/Graz 
Measurement performed by Alicona IF-EdgeMasterModule, 05.09.2025 16:23:58 
